
# Paper 4 — Notebook 05 FINAL: SHAP physical-consistency analysis

This notebook contains **only the final SHAP attribution analysis** used for the
melt-pool-depth manuscript. The obsolete process-map section from the earlier
Notebook 05 is intentionally excluded. Corrected process maps are reproduced by
`Paper4_07_final_validation_AUDITED.ipynb`.

## Purpose

The manuscript uses SHAP as a **post-hoc physical-consistency check**, not as causal
evidence. A model can show physically plausible feature dependence while also using
source-specific correlations.

The analysis:

- checks out the exact MeltpoolNet commit used by the frozen study;
- recreates the frozen regression table and V1 source-study folds when needed;
- uses the F3 depth feature set;
- fits XGBoost on the training portion of one pre-specified source-held-out fold;
- performs inner hyperparameter selection using **training source studies only**;
- computes SHAP values only for the held-out fold;
- saves the feature-importance table, held-out SHAP values, run metadata, and a
  vector PDF figure.

### Pre-specified attribution fold

`SHAP_OUTER_FOLD = 0`

This value is fixed in the code below and must not be changed after viewing SHAP
results simply to obtain a preferred feature ranking.

### Historical manuscript reference

The manuscript currently reports approximately:

- Power: mean |SHAP| = 0.082
- E (J/mm): 0.080
- layer thickness: 0.047
- beam D: 0.035

The final cell prints the reproduced values beside these historical reference values.
If the reproduced values differ materially, **do not force them to match**. Preserve
the notebook output and revise the manuscript or recover the exact legacy analysis
settings before archiving the repository.

No process-map code is included here.


In [ ]:

# ============================================================
# CELL 1 — PINNED, SELF-CONTAINED SETUP
# ============================================================
import os, re, json, time, math, shutil, subprocess, sys, importlib.util, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

PKGS = {
    "pandas":"pandas", "numpy":"numpy", "scikit-learn":"sklearn",
    "pyarrow":"pyarrow", "xgboost":"xgboost", "matplotlib":"matplotlib",
    "scipy":"scipy"
}
missing = [pkg for pkg, imp in PKGS.items() if importlib.util.find_spec(imp) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

from sklearn.model_selection import KFold, GroupKFold, StratifiedKFold, StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    r2_score, mean_absolute_error, f1_score, matthews_corrcoef,
    balanced_accuracy_score, accuracy_score
)
from xgboost import XGBRegressor, XGBClassifier
from scipy.stats import spearmanr

import matplotlib.pyplot as plt

SEED = 42
EXPECTED_COMMIT = "6d68e2acaad8d074134a7f3d843278649774d661"
ART = "/content/artifacts"
RES = "/content/results_08"
REPO = "/content/MeltpoolNet"

os.makedirs(ART, exist_ok=True)
os.makedirs(RES, exist_ok=True)

# ---------- exact source checkout ----------
def ensure_repo():
    if not os.path.exists(os.path.join(REPO, ".git")):
        if os.path.exists(REPO):
            shutil.rmtree(REPO)
        subprocess.run(
            ["git", "clone", "https://github.com/BaratiLab/MeltpoolNet.git", REPO],
            check=True
        )
    subprocess.run(["git", "-C", REPO, "fetch", "--all", "--tags"], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", EXPECTED_COMMIT], check=True)
    commit = subprocess.run(
        ["git", "-C", REPO, "rev-parse", "HEAD"],
        capture_output=True, text=True, check=True
    ).stdout.strip()
    if commit != EXPECTED_COMMIT:
        raise RuntimeError(f"Wrong MeltpoolNet commit: {commit}")
    return commit

commit = ensure_repo()

required = [
    f"{ART}/reg_clean.parquet",
    f"{ART}/cls_clean.parquet",
    f"{ART}/feature_sets.json",
    f"{ART}/meta.json",
]

def artifacts_match_expected():
    if not all(os.path.exists(p) for p in required):
        return False
    try:
        with open(f"{ART}/meta.json") as f:
            m = json.load(f)
        return m.get("data_commit") == EXPECTED_COMMIT
    except Exception:
        return False

# ---------- recreate exact frozen artifacts if absent/stale ----------
if not artifacts_match_expected():
    print("Recreating frozen artifacts from pinned MeltpoolNet commit...")

    reg0 = pd.read_csv(f"{REPO}/Data/meltpoolnet_regression.csv")
    cls0 = pd.read_csv(f"{REPO}/Data/meltpoolnet_classification.csv")

    def drop_junk(df):
        junk = [c for c in df.columns if c.startswith("Unnamed:") or str(c).strip() == ""]
        if "comment" in df.columns:
            junk.append("comment")
        return df.drop(columns=junk)

    reg0 = drop_junk(reg0)
    cls0 = drop_junk(cls0)

    CLASSES0 = ["desirable", "keyhole", "LOF", "balling"]
    CLS_ID0 = {c:i for i,c in enumerate(CLASSES0)}

    cls0 = cls0[cls0["meltpool shape"].isin(CLASSES0)].copy()
    cls0["y_class"] = cls0["meltpool shape"].map(CLS_ID0).astype(int)

    def comp_cols(df):
        return [c for c in df.columns if re.search(r"\(wt\.?%\)", c)]

    reg_F1 = ["Power","Velocity","powder flowrate","layer thickness","beam D","Hatch spacing"]
    reg_F2 = reg_F1 + ["density","Cp","k","melting T","absorption coefficient","minimum absorptivity"]
    reg_F3 = reg_F2 + ["E (J/mm)","E (J/mm3)"]
    reg_F4 = reg_F3 + comp_cols(reg0)
    reg_F5 = reg_F3 + ["Material"]

    cls_F1 = ["Power","Velocity","Hatch spacing","layer thickness","beam D"]
    cls_F2 = cls_F1 + ["density","Cp","k","melting T","absorption coefficient","minimal absorptivity"]
    cls_F3 = cls_F2 + ["p/lb","p/l","p/b2","p/b","vb","vl"]
    cls_F4 = cls_F3 + comp_cols(cls0)
    cls_F5 = cls_F3 + ["Material"]

    REG_FEATURES0 = {"F1":reg_F1,"F2":reg_F2,"F3":reg_F3,"F4":reg_F4,"F5":reg_F5}
    CLS_FEATURES0 = {"F1":cls_F1,"F2":cls_F2,"F3":cls_F3,"F4":cls_F4,"F5":cls_F5}

    def coerce_numeric(df, cols):
        for c in cols:
            if c != "Material" and c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")
        return df

    reg0 = coerce_numeric(reg0, sorted(set(sum(REG_FEATURES0.values(), []))))
    cls0 = coerce_numeric(cls0, sorted(set(sum(CLS_FEATURES0.values(), []))))

    reg0 = reg0.reset_index(drop=True)
    cls0 = cls0.reset_index(drop=True)

    reg0["has_depth"] = reg0["depth of meltpool"].notna()
    reg0["has_width"] = reg0["width of melt pool"].notna()
    reg0["has_class"] = reg0["meltpool shape"].isin(CLASSES0)
    reg0["y_class"] = reg0["meltpool shape"].map(CLS_ID0)

    K_REG, K_CLS = 10, 5

    def reg_splits(mask_col, k=K_REG):
        mask = reg0[mask_col].values
        idx = np.where(mask)[0]
        groups = reg0.loc[idx, "paper ID"].values
        v0 = np.full(len(reg0), -1, dtype=int)
        v1 = np.full(len(reg0), -1, dtype=int)

        for f, (_, te) in enumerate(KFold(k, shuffle=True, random_state=SEED).split(idx)):
            v0[idx[te]] = f
        for f, (_, te) in enumerate(GroupKFold(n_splits=k).split(idx, groups=groups)):
            v1[idx[te]] = f
        return v0, v1

    reg0["v0_depth"], reg0["v1_depth"] = reg_splits("has_depth")
    reg0["v0_width"], reg0["v1_width"] = reg_splits("has_width")

    def lomo_materials(mask_col, min_rows=40):
        vc = reg0.loc[reg0[mask_col], "Material"].value_counts()
        return vc[vc >= min_rows].index.tolist()

    LOMO_DEPTH = lomo_materials("has_depth")
    LOMO_WIDTH = lomo_materials("has_width")

    cy = cls0["y_class"].values
    cg = cls0["paper ID"].values
    cls0["v0"] = -1
    cls0["v1"] = -1

    for f, (_, te) in enumerate(
        StratifiedKFold(K_CLS, shuffle=True, random_state=SEED).split(cls0, cy)
    ):
        cls0.loc[te, "v0"] = f

    for f, (_, te) in enumerate(
        StratifiedGroupKFold(K_CLS, shuffle=True, random_state=SEED).split(cls0, cy, groups=cg)
    ):
        cls0.loc[te, "v1"] = f

    LOMO_CLS = cls0["Material"].value_counts()
    LOMO_CLS = LOMO_CLS[LOMO_CLS >= 40].index.tolist()

    reg0.to_parquet(f"{ART}/reg_clean.parquet", index=False)
    cls0.to_parquet(f"{ART}/cls_clean.parquet", index=False)

    with open(f"{ART}/feature_sets.json", "w") as f:
        json.dump({"regression":REG_FEATURES0, "classification":CLS_FEATURES0}, f, indent=2)

    meta0 = {
        "seed": SEED,
        "source_key": "paper ID",
        "classes": CLASSES0,
        "regression": {
            "primary_target": "depth of meltpool",
            "secondary_target": "width of melt pool",
            "appendix_target": "length of melt pool",
            "k_folds": K_REG,
            "n_depth": int(reg0["has_depth"].sum()),
            "n_width": int(reg0["has_width"].sum()),
            "n_class_labels": int(reg0["has_class"].sum()),
            "n_depth_and_width": int((reg0["has_depth"] & reg0["has_width"]).sum()),
            "lomo_depth_materials": LOMO_DEPTH,
            "lomo_width_materials": LOMO_WIDTH,
        },
        "classification": {
            "k_folds": K_CLS,
            "class_counts": {c:int((cls0["y_class"] == i).sum()) for c,i in CLS_ID0.items()},
            "lomo_materials": LOMO_CLS,
            "note": "Use pooled out-of-fold classification metrics."
        },
        "data_commit": commit,
    }
    with open(f"{ART}/meta.json", "w") as f:
        json.dump(meta0, f, indent=2)

reg = pd.read_parquet(f"{ART}/reg_clean.parquet")
cls = pd.read_parquet(f"{ART}/cls_clean.parquet")
with open(f"{ART}/feature_sets.json") as f:
    FS = json.load(f)
with open(f"{ART}/meta.json") as f:
    META = json.load(f)

if META["data_commit"] != EXPECTED_COMMIT:
    raise RuntimeError("Artifact commit mismatch.")

F3_REG = FS["regression"]["F3"]
F3_CLS = FS["classification"]["F3"]
CLASSES = META["classes"]
CLASS_IDS = np.arange(len(CLASSES))

print("Pinned Paper 4 data ready")
print("  commit                 :", META["data_commit"])
print("  regression rows        :", len(reg))
print("  width-labelled rows    :", int(reg["has_width"].sum()))
print("  depth-labelled rows    :", int(reg["has_depth"].sum()))
print("  classification rows    :", len(cls))
print("  output directory       :", RES)


In [ ]:

# ============================================================
# CELL 2 — FINAL SHAP ANALYSIS ON A PRE-SPECIFIED V1 FOLD
# ============================================================
import importlib.util, subprocess, sys

if importlib.util.find_spec("shap") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "shap"])

import shap
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor

SHAP_OUTER_FOLD = 0

# Depth-labelled frozen table.
d = reg[reg["has_depth"]].reset_index(drop=True).copy()
X = d[F3_REG].copy()
y_raw = pd.to_numeric(d["depth of meltpool"], errors="coerce").values.astype(float)
y = np.log10(y_raw)
groups = d["paper ID"].astype(str).values
fold_ids = d["v1_depth"].astype(int).values

te = fold_ids == SHAP_OUTER_FOLD
tr = (fold_ids != SHAP_OUTER_FOLD) & (fold_ids >= 0)

assert te.sum() > 0
assert tr.sum() > 20
assert set(groups[te]).isdisjoint(set(groups[tr]))

print("Pre-specified V1 attribution fold:", SHAP_OUTER_FOLD)
print("Training rows:", int(tr.sum()))
print("Held-out rows:", int(te.sum()))
print("Training studies:", int(pd.Series(groups[tr]).nunique()))
print("Held-out studies:", int(pd.Series(groups[te]).nunique()))
print("Held-out source IDs:", sorted(pd.Series(groups[te]).unique().tolist()))

# Same compact XGBoost search space used in the final validation workflow.
XGB_GRID = [
    {"max_depth": md, "learning_rate": lr, "subsample": 0.8}
    for md in [3, 6]
    for lr in [0.05, 0.10]
]

def make_xgb(params):
    return XGBRegressor(
        random_state=SEED,
        n_estimators=400,
        max_depth=params["max_depth"],
        learning_rate=params["learning_rate"],
        subsample=params["subsample"],
        verbosity=0,
        n_jobs=-1
    )

def tune_xgb_grouped(Xtr, ytr, gtr):
    splits = list(
        GroupKFold(n_splits=3).split(
            np.arange(len(ytr)), groups=gtr
        )
    )
    best_mae = np.inf
    best_params = None

    for params in XGB_GRID:
        oof = np.full(len(ytr), np.nan)
        for itr, ite in splits:
            mdl = make_xgb(params)
            mdl.fit(Xtr.iloc[itr], ytr[itr])
            oof[ite] = mdl.predict(Xtr.iloc[ite])

        mae = mean_absolute_error(ytr, oof)
        if mae < best_mae:
            best_mae = float(mae)
            best_params = params.copy()

    return best_params, best_mae

best_params, inner_mae = tune_xgb_grouped(
    X.loc[tr].reset_index(drop=True),
    y[tr],
    groups[tr]
)

model = make_xgb(best_params)
model.fit(X.loc[tr], y[tr])

pred = model.predict(X.loc[te])
heldout_r2 = float(r2_score(y[te], pred))
heldout_mae_log = float(mean_absolute_error(y[te], pred))

print("\nSelected XGBoost parameters:", best_params)
print("Inner grouped MAE(log10):", round(inner_mae, 5))
print("Held-out fold R2(log10):", round(heldout_r2, 5))
print("Held-out fold MAE(log10):", round(heldout_mae_log, 5))

# TreeSHAP on held-out rows only.
explainer = shap.TreeExplainer(model)
sv = explainer.shap_values(X.loc[te])

if isinstance(sv, list):
    sv = sv[0]
sv = np.asarray(sv)

if sv.shape != (int(te.sum()), len(F3_REG)):
    raise RuntimeError(
        f"Unexpected SHAP shape {sv.shape}; expected {(int(te.sum()), len(F3_REG))}"
    )

importance = pd.DataFrame({
    "feature": F3_REG,
    "mean_abs_SHAP_log10_depth": np.mean(np.abs(sv), axis=0)
}).sort_values("mean_abs_SHAP_log10_depth", ascending=False)

importance.to_csv(f"{RES}/shap_feature_importance_fold0.csv", index=False)

held = d.loc[te, ["paper ID", "Material", "depth of meltpool"]].reset_index(drop=True)
held["pred_log10_depth"] = pred
held["true_log10_depth"] = y[te]

for j, feature in enumerate(F3_REG):
    held[f"SHAP::{feature}"] = sv[:, j]

held.to_csv(f"{RES}/shap_values_fold0.csv", index=False)

metadata = {
    "data_commit": META.get("data_commit"),
    "outer_protocol": "V1 source-study-held-out",
    "outer_fold": SHAP_OUTER_FOLD,
    "n_train_rows": int(tr.sum()),
    "n_test_rows": int(te.sum()),
    "n_train_studies": int(pd.Series(groups[tr]).nunique()),
    "n_test_studies": int(pd.Series(groups[te]).nunique()),
    "heldout_source_ids": sorted(pd.Series(groups[te]).unique().tolist()),
    "target": "log10(depth of meltpool)",
    "feature_set": "F3",
    "model": "XGBoost",
    "best_params": best_params,
    "inner_grouped_MAE_log10": inner_mae,
    "heldout_R2_log10": heldout_r2,
    "heldout_MAE_log10": heldout_mae_log,
    "interpretation": (
        "Post-hoc physical-consistency check only; SHAP does not establish "
        "causality or exclude source-specific confounding."
    )
}

with open(f"{RES}/shap_run_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("\nTop SHAP features:")
print(importance.head(10).to_string(index=False))


In [ ]:

# ============================================================
# CELL 3 — MANUSCRIPT-STYLE SHAP FIGURE
# ============================================================
power_idx = F3_REG.index("Power")
power_x = pd.to_numeric(X.loc[te, "Power"], errors="coerce").values
power_shap = sv[:, power_idx]

top = importance.head(10).sort_values(
    "mean_abs_SHAP_log10_depth", ascending=True
)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))

ax[0].barh(
    top["feature"],
    top["mean_abs_SHAP_log10_depth"]
)
ax[0].set_xlabel("mean |SHAP| (log10 depth units)")
ax[0].set_title("(a) Feature importance, unseen-study fold")
ax[0].grid(axis="x", alpha=0.3)

ok = np.isfinite(power_x) & np.isfinite(power_shap)
ax[1].scatter(power_x[ok], power_shap[ok], s=20, alpha=0.7)
ax[1].axhline(0, linestyle="--", linewidth=1)
ax[1].set_xlabel("Power")
ax[1].set_ylabel("SHAP value for Power")
ax[1].set_title("(b) Dependence: Power")
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{RES}/fig_shap_final.pdf", bbox_inches="tight")
plt.savefig(f"{RES}/fig_shap_final.png", dpi=600, bbox_inches="tight")
plt.show()

print("Saved:")
print(f"  {RES}/fig_shap_final.pdf")
print(f"  {RES}/fig_shap_final.png")


In [ ]:

# ============================================================
# CELL 4 — REPRODUCTION AUDIT + PACKAGE
# ============================================================
historical = {
    "Power": 0.082,
    "E (J/mm)": 0.080,
    "layer thickness": 0.047,
    "beam D": 0.035,
}

obs = importance.set_index("feature")["mean_abs_SHAP_log10_depth"]

rows = []
for feature, ref in historical.items():
    value = float(obs.get(feature, np.nan))
    rows.append({
        "feature": feature,
        "historical_manuscript_reference": ref,
        "reproduced_value": value,
        "absolute_difference": abs(value - ref) if np.isfinite(value) else np.nan,
    })

audit = pd.DataFrame(rows)
audit.to_csv(f"{RES}/shap_historical_reference_check.csv", index=False)

print("Historical-reference check")
print(audit.round(4).to_string(index=False))

print(
    "\nIMPORTANT: The historical values are only a cross-check. "
    "If this clean, pre-specified analysis produces materially different values, "
    "do not alter the fold/model after looking at the result. Preserve the output "
    "and revise the manuscript or recover the exact legacy settings."
)

archive = "/content/Paper4_05_SHAP_FINAL_results"
shutil.make_archive(archive, "zip", RES)

print("\nCreated ZIP:")
print(archive + ".zip")
